# Olist Brazilian E-Commerce — Phase 1: ExplorationUnderstanding the 9 tables before touching analysis. Output: clean master DataFrame for downstream phases.**Goals for this phase:**1. Shape, schema, types across all tables2. Join integrity — do foreign keys line up?3. Missingness patterns and their business meaning4. Temporal coverage and anomalies5. Order status distribution → analytical filter strategy6. Cardinality of one-to-many relationships → aggregation rules7. Build and persist the order-level master DataFrame for Phase 2+

## Imports and setup

In [12]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')

print(f"Raw data folder exists: {RAW.exists()}")
print(f"Processed folder exists: {PROCESSED.exists()}")
print(f"Files in raw: {len(list(RAW.glob('*.csv')))}")

Raw data folder exists: True
Processed folder exists: True
Files in raw: 9


## Loading the 9 tables

In [ ]:
tables = {
    'customers':   pd.read_csv(RAW / 'olist_customers_dataset.csv'),
    'orders':      pd.read_csv(RAW / 'olist_orders_dataset.csv'),
    'order_items': pd.read_csv(RAW / 'olist_order_items_dataset.csv'),
    'payments':    pd.read_csv(RAW / 'olist_order_payments_dataset.csv'),
    'reviews':     pd.read_csv(RAW / 'olist_order_reviews_dataset.csv'),
    'products':    pd.read_csv(RAW / 'olist_products_dataset.csv'),
    'sellers':     pd.read_csv(RAW / 'olist_sellers_dataset.csv'),
    'geolocation': pd.read_csv(RAW / 'olist_geolocation_dataset.csv'),
    'categories':  pd.read_csv(RAW / 'product_category_name_translation.csv'),
}

summary = pd.DataFrame([
    {
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'memory_mb': round(df.memory_usage(deep=True).sum() / 1024**2, 1)
    }
    for name, df in tables.items()
])

print(summary.to_string(index=False))
print(f"\nTotal memory: {summary['memory_mb'].sum():.1f} MB")

      table    rows  columns  memory_mb
  customers   99441        5       11.0
     orders   99441        8       21.9
order_items  112650        7       18.4
   payments  103886        5        8.1
    reviews   99224        7       17.8
   products   32951        9        3.7
    sellers    3095        4        0.2
geolocation 1000163        5       50.1
 categories      71        2        0.0

Total memory: 131.2 MB


: 

## Schema inspectionSurveying every column's type and non-null count in one pass. This is where we spot columns that loaded as strings instead of dates, and where missingness lives.

In [ ]:
for name, df in tables.items():
    print(f"\n{'='*60}")
    print(f"TABLE: {name}  ({len(df):,} rows × {df.shape[1]} cols)")
    print('='*60)
    df.info(show_counts=True)


TABLE: customers  (99,441 rows × 5 cols)
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB

TABLE: orders  (99,441 rows × 8 cols)
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purcha

: 

## Parsing date columnsAll date and timestamp columns loaded as strings. Converting them explicitly (rather than via `parse_dates` at read time) keeps the transformation visible in the notebook.

In [ ]:
date_cols = {
    'orders': [
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
    ],
    'order_items': ['shipping_limit_date'],
    'reviews': ['review_creation_date', 'review_answer_timestamp'],
}

for table_name, cols in date_cols.items():
    for col in cols:
        tables[table_name][col] = pd.to_datetime(
            tables[table_name][col], errors='coerce'
        )

for table_name, cols in date_cols.items():
    print(f"\n{table_name}:")
    print(tables[table_name][cols].dtypes)


orders:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

order_items:
shipping_limit_date    datetime64[us]
dtype: object

reviews:
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


: 

## Temporal coverageKaggle claims 2016–2018 coverage. We verify exact boundaries and look for partial-month edge effects and anomalies before trusting any time-series analysis.

In [ ]:
orders = tables['orders']

print("PURCHASE DATE RANGE")
print(f"  Earliest: {orders['order_purchase_timestamp'].min()}")
print(f"  Latest:   {orders['order_purchase_timestamp'].max()}")
print(f"  Span:     {(orders['order_purchase_timestamp'].max() - orders['order_purchase_timestamp'].min()).days} days")

monthly = (
    orders
    .assign(year_month=orders['order_purchase_timestamp'].dt.to_period('M'))
    .groupby('year_month')
    .size()
    .reset_index(name='orders')
)
monthly['year_month'] = monthly['year_month'].astype(str)

print(f"\nMONTHLY ORDER COUNTS ({len(monthly)} months)")
print(monthly.to_string(index=False))

PURCHASE DATE RANGE
  Earliest: 2016-09-04 21:15:19
  Latest:   2018-10-17 17:30:18
  Span:     772 days

MONTHLY ORDER COUNTS (25 months)
year_month  orders
   2016-09       4
   2016-10     324
   2016-12       1
   2017-01     800
   2017-02    1780
   2017-03    2682
   2017-04    2404
   2017-05    3700
   2017-06    3245
   2017-07    4026
   2017-08    4331
   2017-09    4285
   2017-10    4631
   2017-11    7544
   2017-12    5673
   2018-01    7269
   2018-02    6728
   2018-03    7211
   2018-04    6939
   2018-05    6873
   2018-06    6167
   2018-07    6292
   2018-08    6512
   2018-09      16
   2018-10       4


: 

**Key findings:**- **Analytical window: Jan 2017 – Aug 2018** (20 full months). Partial edge months (Sep 2016, Dec 2016, Sep/Oct 2018 — 349 orders total, <0.4%) will be excluded from time-series analysis.- **November 2017 spike** (7,544 orders, +63% vs October) = Brazil Black Friday. Real business signal, not noise.

## Order status distributionBefore analyzing anything about orders, we need to know what kinds of orders exist. This drives our filtering strategy for hypothesis tests.

In [ ]:
status_counts = orders['order_status'].value_counts()
status_pct = (status_counts / len(orders) * 100).round(2)

status_summary = pd.DataFrame({
    'count': status_counts,
    'pct': status_pct
})
print("ORDER STATUS DISTRIBUTION")
print(status_summary.to_string())
print(f"\nTotal orders: {len(orders):,}")

ORDER STATUS DISTRIBUTION
              count    pct
order_status              
delivered     96478  97.02
shipped        1107   1.11
canceled        625   0.63
unavailable     609   0.61
invoiced        314   0.32
processing      301   0.30
created           5   0.01
approved          2   0.00

Total orders: 99,441


: 

**Methodological decision:** For delivery-time / satisfaction / freight analyses we filter to `status == 'delivered'` only (96,478 orders, 97.0%). For payment and order-volume analyses we keep all statuses — canceled orders still tell us something about behavior.The 2,965 missing `delivered_customer_date` values align almost exactly with the 2,963 non-delivered orders. Missingness isn't random — it's explained by order status.

## Join integrityVerify that foreign keys actually line up before denormalizing. Orphan records would silently disappear in joins.

In [ ]:
joins_to_check = [
    ('orders',      'customer_id',  'customers',   'customer_id'),
    ('order_items', 'order_id',     'orders',      'order_id'),
    ('order_items', 'product_id',   'products',    'product_id'),
    ('order_items', 'seller_id',    'sellers',     'seller_id'),
    ('payments',    'order_id',     'orders',      'order_id'),
    ('reviews',     'order_id',     'orders',      'order_id'),
]

print("JOIN INTEGRITY CHECK")
print("="*75)
print(f"{'left_table':<14} {'left_key':<14} → {'right_table':<12} {'right_key':<14} {'orphans':>8}")
print("-"*75)

for left_tbl, left_key, right_tbl, right_key in joins_to_check:
    left_values = set(tables[left_tbl][left_key])
    right_values = set(tables[right_tbl][right_key])
    orphans = len(left_values - right_values)
    orphan_pct = orphans / len(left_values) * 100
    flag = "OK" if orphans == 0 else "WARN"
    print(f"{left_tbl:<14} {left_key:<14} → {right_tbl:<12} {right_key:<14} {orphans:>6} {flag}  ({orphan_pct:.2f}%)")

JOIN INTEGRITY CHECK
left_table     left_key       → right_table  right_key       orphans
---------------------------------------------------------------------------
orders         customer_id    → customers    customer_id         0 OK  (0.00%)
order_items    order_id       → orders       order_id            0 OK  (0.00%)
order_items    product_id     → products     product_id          0 OK  (0.00%)
order_items    seller_id      → sellers      seller_id           0 OK  (0.00%)
payments       order_id       → orders       order_id            0 OK  (0.00%)
reviews        order_id       → orders       order_id            0 OK  (0.00%)


: 

Perfect referential integrity — zero orphans across all six relationships. Joins will be lossless.

## Cardinality — one-to-many relationshipsBefore denormalizing, we need to know how many payments, items, and reviews attach to a single order. This drives the aggregation rules when collapsing to order-level.

In [ ]:
print("CARDINALITY PER ORDER")
print("="*60)

for table_name, key in [('order_items', 'order_id'),
                         ('payments',   'order_id'),
                         ('reviews',    'order_id')]:
    counts = tables[table_name].groupby(key).size()
    print(f"\n{table_name} per order:")
    print(f"  orders with records:  {len(counts):,}")
    print(f"  mean per order:       {counts.mean():.2f}")
    print(f"  median per order:     {counts.median():.0f}")
    print(f"  max per order:        {counts.max()}")
    print(f"  distribution:")
    print(counts.value_counts().sort_index().head(10).to_string())

CARDINALITY PER ORDER

order_items per order:
  orders with records:  98,666
  mean per order:       1.14
  median per order:     1
  max per order:        21
  distribution:
1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8

payments per order:
  orders with records:  99,440
  mean per order:       1.04
  median per order:     1
  max per order:        29
  distribution:
1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5

reviews per order:
  orders with records:  98,673
  mean per order:       1.01
  median per order:     1
  max per order:        3
  distribution:
1    98126
2      543
3        4


: 

**Aggregation rules locked in:**| Source | Rule when collapsing to order-level ||---|---|| `order_items` | SUM price, SUM freight, COUNT items || `payments` | SUM value, COUNT payments, FIRST type (dominant), MAX installments || `reviews` | LATEST review_score (handles the 547 orders with multiple reviews) |The "first payment_type" choice is a defensible simplification — the alternative (one row per payment) would over-count multi-payment orders in hypothesis tests.

## Building the order-level master DataFrameCollapse the 9 tables into one denormalized DataFrame where one row = one order. This is the artifact Phases 2, 3, and 4 will use.

In [ ]:
# Aggregate order_items to order-level
items_agg = (
    tables['order_items']
    .groupby('order_id')
    .agg(
        items_count=('order_item_id', 'count'),
        items_price_total=('price', 'sum'),
        freight_total=('freight_value', 'sum'),
    )
    .reset_index()
)

# Aggregate payments to order-level
payments_agg = (
    tables['payments']
    .sort_values(['order_id', 'payment_sequential'])
    .groupby('order_id')
    .agg(
        payment_value_total=('payment_value', 'sum'),
        payments_count=('payment_sequential', 'count'),
        payment_type=('payment_type', 'first'),
        payment_installments_max=('payment_installments', 'max'),
    )
    .reset_index()
)

# Aggregate reviews to order-level (latest review per order)
reviews_agg = (
    tables['reviews']
    .sort_values(['order_id', 'review_creation_date'])
    .groupby('order_id')
    .agg(
        review_score=('review_score', 'last'),
        review_creation_date=('review_creation_date', 'last'),
    )
    .reset_index()
)

# Start from orders, merge customer info
master = tables['orders'].merge(
    tables['customers'][['customer_id', 'customer_unique_id', 'customer_state', 'customer_city']],
    on='customer_id',
    how='left'
)

# Left-join the three aggregates
master = (
    master
    .merge(items_agg,    on='order_id', how='left')
    .merge(payments_agg, on='order_id', how='left')
    .merge(reviews_agg,  on='order_id', how='left')
)

print(f"Original orders rows: {len(tables['orders']):,}")
print(f"Master DataFrame rows: {len(master):,}")
print(f"Row count preserved: {len(master) == len(tables['orders'])}")
print(f"\nMaster DataFrame columns ({master.shape[1]}):")
print(master.columns.tolist())

Original orders rows: 99,441
Master DataFrame rows: 99,441
Row count preserved: True

Master DataFrame columns (20):
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_state', 'customer_city', 'items_count', 'items_price_total', 'freight_total', 'payment_value_total', 'payments_count', 'payment_type', 'payment_installments_max', 'review_score', 'review_creation_date']


: 

## Derived analytical columnsBuild the measures our hypothesis tests need — delivery duration, on-time flag, approval lag — plus English product category for category-based tests. Computed once in the master, reused everywhere.**Category handling:** An order can contain multiple products from different categories. We use the category of the most-expensive item as the order's dominant category — it's the best single-category proxy since the expensive item dominates the purchase decision.**On-time definition:** `delivered_customer_date <= estimated_delivery_date`. This uses Olist's own delivery promise, not an arbitrary threshold.

In [ ]:
# Time-based measures
master['delivery_days'] = (
    master['order_delivered_customer_date'] - master['order_purchase_timestamp']
).dt.total_seconds() / 86400

master['approval_lag_hours'] = (
    master['order_approved_at'] - master['order_purchase_timestamp']
).dt.total_seconds() / 3600

master['estimated_vs_actual_days'] = (
    master['order_estimated_delivery_date'] - master['order_delivered_customer_date']
).dt.total_seconds() / 86400

master['on_time'] = master['estimated_vs_actual_days'] >= 0

# Dominant category = category of highest-priced item in the order
dominant_category = (
    tables['order_items']
    .sort_values(['order_id', 'price'], ascending=[True, False])
    .drop_duplicates('order_id', keep='first')
    [['order_id', 'product_id']]
    .merge(tables['products'][['product_id', 'product_category_name']], on='product_id', how='left')
    .merge(tables['categories'], on='product_category_name', how='left')
    .rename(columns={'product_category_name_english': 'category'})
    [['order_id', 'category']]
)

master = master.merge(dominant_category, on='order_id', how='left')

# Month column for time-series filtering and plotting
master['purchase_month'] = master['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Sanity check
print(f"Master DataFrame: {len(master):,} rows × {master.shape[1]} columns\n")

print("NEW COLUMNS — null check")
print("="*60)
for col in ['delivery_days', 'approval_lag_hours', 'estimated_vs_actual_days', 'on_time', 'category', 'purchase_month']:
    null_count = master[col].isna().sum()
    null_pct = null_count / len(master) * 100
    print(f"  {col:<30} nulls: {null_count:>6,}  ({null_pct:5.2f}%)")

print("\nQUICK STATS — delivered orders only")
print("="*60)
delivered = master[master['order_status'] == 'delivered']
print(f"Delivered orders:           {len(delivered):,}")
print(f"Mean delivery days:         {delivered['delivery_days'].mean():.2f}")
print(f"Median delivery days:       {delivered['delivery_days'].median():.2f}")
print(f"On-time rate:               {delivered['on_time'].mean()*100:.2f}%")
print(f"Mean review score:          {delivered['review_score'].mean():.2f}")

Master DataFrame: 99,441 rows × 26 columns

NEW COLUMNS — null check
  delivery_days                  nulls:  2,965  ( 2.98%)
  approval_lag_hours             nulls:    160  ( 0.16%)
  estimated_vs_actual_days       nulls:  2,965  ( 2.98%)
  on_time                        nulls:      0  ( 0.00%)
  category                       nulls:  2,212  ( 2.22%)
  purchase_month                 nulls:      0  ( 0.00%)

QUICK STATS — delivered orders only
Delivered orders:           96,478
Mean delivery days:         12.56
Median delivery days:       10.22
On-time rate:               91.88%
Mean review score:          4.16


: 

## Fixing nullable boolean, then persisting to parquet`on_time` was computed as `estimated_vs_actual_days >= 0`. For non-delivered orders (where `delivery_days` is null), this evaluated to `False` — misleading, since those orders weren't late, they were never delivered. We convert to a **nullable boolean** (pandas' `'boolean'` dtype) and set the 2,965 non-delivered orders to `pd.NA`.**Why parquet over CSV for the processed file:** preserves datetime and nullable types exactly, ~10x faster to load, smaller on disk.

In [ ]:
# Convert on_time to nullable boolean so it can hold NaN
master['on_time'] = master['on_time'].astype('boolean')
master.loc[master['delivery_days'].isna(), 'on_time'] = pd.NA

print("AFTER FIX")
print(f"on_time dtype: {master['on_time'].dtype}")
print(f"on_time value distribution:")
print(master['on_time'].value_counts(dropna=False))
print(f"\non_time nulls: {master['on_time'].isna().sum():,} (should be 2,965)")

# Save to parquet
output_path = PROCESSED / 'orders_master.parquet'
master.to_parquet(output_path, index=False)

import os
file_size_mb = os.path.getsize(output_path) / 1024**2
reloaded = pd.read_parquet(output_path)

print(f"\nSAVED")
print(f"  Path: {output_path}")
print(f"  Size: {file_size_mb:.2f} MB")
print(f"  Reload verification: {len(reloaded):,} rows × {reloaded.shape[1]} cols")
print(f"  on_time dtype after reload: {reloaded['on_time'].dtype}")

AFTER FIX
on_time dtype: boolean
on_time value distribution:
on_time
True     88649
False     7827
<NA>      2965
Name: count, dtype: Int64

on_time nulls: 2,965 (should be 2,965)

SAVED
  Path: ..\data\processed\orders_master.parquet
  Size: 16.11 MB
  Reload verification: 99,441 rows × 26 cols
  on_time dtype after reload: boolean


: 

### Phase 1 — Findings Summary# Dataset overview- **99,441 orders** spanning September 2016 – October 2018- **9 normalized tables**, 126 MB raw, 131 MB in memory- **Perfect referential integrity** — zero orphan keys across all 6 foreign-key relationships### Temporal boundaries- **Analytical window: Jan 2017 – Aug 2018** (20 full months, ~97k orders)- 349 orders in partial edge months (Sep 2016, Dec 2016, Sep/Oct 2018) excluded from time-series analysis- **November 2017 spike** (7,544 orders, +63% vs October) = Brazil Black Friday### Order status distribution- **97.0% delivered** → analytical backbone for delivery/satisfaction hypothesis tests- 3.0% intermediate or failed (shipped, canceled, unavailable, invoiced, processing)- **Decision**: `status == 'delivered'` filter required for any delivery-time or satisfaction analysis### Data quality flags- **2,965 missing `delivered_customer_date`** — explained entirely by non-delivered status (not random nulls)- **2,212 missing categories** (~2.2%) — mix of products without categories + orders without items- **1 order with no payment record** — real data quality anomaly, flagged- **775 orders with no items** — aligns with canceled/unavailable population- **Reviews are score-complete** (100%) but comment-sparse (title: 12%, message: 41%) — NLP out of scope, scores usable### Cardinality / fan-out patterns- Most orders: 1 item, 1 payment, 1 review- **Max payments per order: 29** — Brazil's installment culture creates payment sequences- **Aggregation rules locked**: sum for prices/values, first/max for categoricals, latest for reviews### Master DataFrame- **99,441 rows × 26 columns** at order level — row count preserved through all joins- Derived columns: `delivery_days`, `approval_lag_hours`, `estimated_vs_actual_days`, `on_time`, `category`, `purchase_month`- **`on_time` uses nullable boolean** — null when never delivered (3% of orders)- Saved to `data/processed/orders_master.parquet` (16 MB) for Phase 2+### Baseline business metrics (delivered orders)| Metric | Value ||---|---|| Mean delivery days | 12.56 || Median delivery days | 10.22 || On-time rate | 91.88% || Mean review score | 4.16 / 5 |### Implications for Phase 2+- **Hypothesis test on delivery → satisfaction** is viable: clean `delivery_days` + `review_score` pair on 96,478 delivered orders- **Category → order value (ANOVA)**: usable on the 97.8% of orders with known category- **Payment type → completion**: definable using `order_status == 'delivered'` as success proxy; `credit_card` vs `boleto` as main groups- **Black Friday (Nov 2017)** may need separate analysis or exclusion in baseline tests — it's a structural outlier